# 🔥 Cohort Analysis — Customer Retention

> **Dataset:** Online Retail (UCI Machine Learning Repository)  
> **Source:** https://archive.ics.uci.edu/dataset/352/online+retail  
> **Goal:** Measure how well a UK-based e-commerce store retains customers over time, identify the critical churn window, and surface actionable recommendations.

---

## Business Question

> *"Of the customers who made their first purchase in a given month, how many came back in the following months?"*

Cohort analysis answers this by grouping customers by their **acquisition month** and tracking retention over time.

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Optional: download dataset automatically
# pip install openpyxl

print('Libraries loaded ✅')

## 2. Load the Data

Download the dataset from UCI: https://archive.ics.uci.edu/dataset/352/online+retail  
Save it as `online_retail.xlsx` in the `data/` folder.

In [ ]:
df = pd.read_excel('data/online_retail.xlsx', dtype={'CustomerID': str})

print(f'Shape: {df.shape}')
print(f'Date range: {df.InvoiceDate.min().date()} → {df.InvoiceDate.max().date()}')
df.head()

## 3. Data Cleaning

In [ ]:
raw_rows = len(df)

# Remove cancelled orders (InvoiceNo starts with 'C')
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# Remove rows without CustomerID
df = df.dropna(subset=['CustomerID'])

# Remove negative quantity or price
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# Compute revenue
df['Revenue'] = df['Quantity'] * df['UnitPrice']

print(f'Rows removed: {raw_rows - len(df):,}')
print(f'Clean rows:   {len(df):,}')
print(f'Unique customers: {df.CustomerID.nunique():,}')

## 4. Build Cohort Table

In [ ]:
# Convert to period months
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')

# Get each customer's FIRST purchase month (cohort)
df['CohortMonth'] = df.groupby('CustomerID')['InvoiceMonth'].transform('min')

# Calculate cohort index (months since first purchase)
df['CohortIndex'] = (
    df['InvoiceMonth'].dt.to_timestamp() -
    df['CohortMonth'].dt.to_timestamp()
).dt.days // 30

df[['CustomerID', 'InvoiceMonth', 'CohortMonth', 'CohortIndex']].drop_duplicates().head(10)

In [ ]:
# Count unique customers per cohort/period
cohort_data = (
    df.groupby(['CohortMonth', 'CohortIndex'])['CustomerID']
    .nunique()
    .reset_index()
)

# Pivot to matrix
cohort_pivot = cohort_data.pivot_table(
    index='CohortMonth',
    columns='CohortIndex',
    values='CustomerID'
)

# Retention rates (relative to cohort size at month 0)
cohort_size = cohort_pivot.iloc[:, 0]
retention = cohort_pivot.divide(cohort_size, axis=0).round(3)

print(f'Cohort matrix: {retention.shape[0]} cohorts × {retention.shape[1]} periods')
retention.head()

## 5. Retention Heatmap

The core visualisation: each cell shows what % of customers from a given cohort were still active N months later.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Custom dark palette
cmap = sns.color_palette('YlGn', as_cmap=True)

sns.heatmap(
    retention,
    annot=True,
    fmt='.0%',
    cmap=cmap,
    linewidths=0.4,
    linecolor='#1a1a1a',
    ax=ax,
    cbar_kws={'label': 'Retention Rate', 'shrink': 0.8},
    annot_kws={'size': 9}
)

ax.set_title('Customer Retention by Cohort', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Months Since First Purchase', fontsize=11, labelpad=10)
ax.set_ylabel('Acquisition Cohort', fontsize=11, labelpad=10)
ax.tick_params(axis='x', rotation=0)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('outputs/cohort_retention_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#fafafa')
plt.show()
print('Chart saved to outputs/ ✅')

## 6. Average Retention Curve

Averaging across all cohorts reveals the **typical customer journey** and the critical drop-off moment.

In [ ]:
avg_retention = retention.mean().reset_index()
avg_retention.columns = ['Month', 'Avg_Retention']

fig, ax = plt.subplots(figsize=(12, 5))

ax.fill_between(avg_retention['Month'], avg_retention['Avg_Retention'],
                alpha=0.15, color='#4ade80')
ax.plot(avg_retention['Month'], avg_retention['Avg_Retention'],
        color='#4ade80', linewidth=2.5, marker='o', markersize=6)

# Annotate month 1 drop
m0 = avg_retention.loc[avg_retention['Month']==0, 'Avg_Retention'].values[0]
m1 = avg_retention.loc[avg_retention['Month']==1, 'Avg_Retention'].values[0]
drop = (m0 - m1) / m0
ax.annotate(
    f'Critical drop: −{drop:.0%}\nat month 1',
    xy=(1, m1), xytext=(2.5, m1 + 0.08),
    arrowprops=dict(arrowstyle='->', color='#f87171'),
    color='#f87171', fontsize=10
)

ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_xlabel('Months Since First Purchase', fontsize=11)
ax.set_ylabel('Average Retention Rate', fontsize=11)
ax.set_title('Average Customer Retention Curve', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_facecolor('#f8faf8')
fig.patch.set_facecolor('#fafafa')

plt.tight_layout()
plt.savefig('outputs/avg_retention_curve.png', dpi=150, bbox_inches='tight', facecolor='#fafafa')
plt.show()

## 7. Cohort Revenue Analysis

Not all retained customers are equal. Here we look at **revenue per active customer** per cohort.

In [ ]:
revenue_data = (
    df.groupby(['CohortMonth', 'CohortIndex'])['Revenue']
    .sum()
    .reset_index()
)

revenue_pivot = revenue_data.pivot_table(
    index='CohortMonth',
    columns='CohortIndex',
    values='Revenue'
)

# Revenue per customer in cohort
revenue_per_customer = revenue_pivot.divide(cohort_size, axis=0).round(2)

fig, ax = plt.subplots(figsize=(16, 8))

sns.heatmap(
    revenue_per_customer,
    annot=True,
    fmt='.0f',
    cmap='Blues',
    linewidths=0.4,
    linecolor='#e0e0e0',
    ax=ax,
    cbar_kws={'label': 'Avg Revenue per Customer (£)', 'shrink': 0.8},
    annot_kws={'size': 8}
)

ax.set_title('Average Revenue per Customer by Cohort (£)', fontsize=15, fontweight='bold', pad=20)
ax.set_xlabel('Months Since First Purchase', fontsize=11)
ax.set_ylabel('Acquisition Cohort', fontsize=11)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('outputs/cohort_revenue_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#fafafa')
plt.show()

## 8. Key Findings & Recommendations

---

### 🔍 Findings

| # | Finding |
|---|---|
| 1 | **Largest drop occurs at Month 1** — typically 60–70% of first-time buyers never return |
| 2 | **Retention stabilises after Month 3** — customers who survive 3 months are significantly more loyal |
| 3 | **Q4 cohorts (Nov/Dec) show lower retention** — likely seasonal buyers with no long-term intent |
| 4 | **Revenue per customer grows** for retained cohorts — strong upsell signal for loyal customers |

---

### 💡 Recommendations

**1. Win-back campaign in Month 1**  
The biggest retention opportunity is converting single-purchase buyers. A targeted email sequence at Day 14 and Day 30 post-purchase could materially improve Month 1 retention.

**2. "Month 3" loyalty milestone**  
Customers who make it to Month 3 have significantly higher LTV. Introduce a loyalty reward or exclusive offer triggered at this point.

**3. Segment seasonal buyers**  
Q4 cohorts behave differently. Marketing spend targeting these customers in Q1 may have poor ROI — filter them out of re-engagement campaigns.

**4. High-value retained cohorts = upsell opportunity**  
Revenue per customer increases for retained cohorts. Prioritise these segments for upsell and premium product campaigns.

---

*Analysis by Danai Avratoglou | Data: UCI Online Retail Dataset*